# Farm boundaries from PlanetScope: train, improve, compare, map (Kaggle, end to end) — round 2

One run does everything: preprocess → train several improved models → compare them fairly → test the winner
once → map every scene → write a report with training curves, confusion matrices and scores by farm size.

> **Keep your outputs:** use **Save Version → Save & Run All (Commit)**. Only a *committed* version keeps
> `/kaggle/working` (an interactive session loses it when it stops). The notebook also writes
> `results_backup.zip` after every experiment, and, if you add the secret below, uploads the models to a
> private Kaggle dataset as it goes.

**What changed since round 1** (round 1, validation: best object F1 0.28 = ResNet-50, 1-px boundaries; most
errors were farms *split* into pieces; threshold tuning ended on the edges of its grid):

| Improvement | Why |
|---|---|
| **Distance-to-boundary head** (ResUNet-a's third task) | each farm's predicted distance map peaks once, in its core: one clean seed per farm, so fewer split farms |
| **Wider, two-stage post-processing search** (seed source, seed / extent / boundary thresholds, minimum seed and farm size) | round 1's best values sat on the grid edges |
| **Test-time augmentation** (average over flips) | smoother, more consistent probabilities |
| **Ensemble of the two best models** (kept only if it beats the best single model on validation) | errors of different models partly cancel |
| **Two GPUs** (T4 x2) and a **time budget** per experiment | more epochs in the 12 h limit; an experiment that runs out of time resumes next version |
| Dropped: 6 m boundaries, training on traced farms only | both were clearly worse in round 1 |

**Inputs:** *Add Input → Your Work → Datasets →* `laikipia-planetscope` (private). Code + farm polygons come from
GitHub. **Settings:** *GPU T4 x2*, *Internet On*. Optional backup: *Add-ons → Secrets →* `KAGGLE_API_TOKEN`
(your Kaggle API token) and set `BACKUP_DATASET` below.

## 1. Settings

In [ ]:
from pathlib import Path

DATASET_NAME = "laikipia-planetscope"      # private dataset with the Planet scenes (+ UDM2)
REPO = "https://github.com/mangaorphy/Farm_Digitisation.git"
BRANCH = "main"
PROJECT_SUBDIR = "farm_boundary_detection"
KAGGLE_USER = "orpheusmanga"
BACKUP_DATASET = "laikipia-farm-models"    # private dataset for model backups ("" = no upload backup)

QUICK_TEST = False        # True = ~20 min dry run of the whole notebook on a few patches
TIME_BUDGET_H = 11.0      # whole notebook; training gets what is left after preprocessing / evaluation reserves
EVAL_RESERVE_H = 1.5      # kept for comparison, test and mapping
EPOCHS = 60               # upper limit per experiment (early stopping usually ends earlier)
PATIENCE = 10
USE_TTA = True            # flip averaging for validation, test and mapping
ENSEMBLE_TOP = 2          # ensemble of the N best single models (0 = no ensemble)
MAP_ALL_SCENES = True

# Preprocessing variants. The reference variant only provides the common yardstick (hand-traced farms).
PREP = {
    "b3_all":    ["masks.boundary_width_m=3.0"],
    "b3_traced": ["masks.boundary_width_m=3.0", "labels.keep={QA_FLAG: [OK], SOURCE: [Digitized]}"],
}
REFERENCE_PREP = "b3_traced"

R50 = ["model.encoder=resnet50", "train.batch_size=12"]   # batch over 2 GPUs
DIST = ["model.distance_head=true"]
# experiment -> (preprocessing variant, overrides); order = simplest first (wins ties)
EXPERIMENTS = {
    "E1_r50":          ("b3_all", R50),                                               # round-1 winner (baseline)
    "E2_r50_dist":     ("b3_all", R50 + DIST),                                        # + distance head
    "E3_r50_dist_bnd": ("b3_all", R50 + DIST + ["loss.lambda_boundary=3.0",           # + stronger boundary loss
                                                "loss.boundary.tversky=1.0"]),
    "E4_r34_dist":     ("b3_all", ["model.encoder=resnet34", "train.batch_size=16"] + DIST),  # lighter, for the ensemble
}

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
import time
T0 = time.time()

## 2. Install and get the code

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio", "geopandas", "pyogrio", "scikit-image",
                "kaggle>=1.7"], check=True)
code_dir = WORK / "Farm_Digitisation"
if not code_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(code_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(code_dir), "pull", "--ff-only"], check=False)
PROJECT = code_dir / PROJECT_SUBDIR
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
import torch, src
print("code", src.__version__, "| torch", torch.__version__, "| GPUs:",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "NONE - turn on the GPU!")

## 3. Imagery, configuration and backups

In [ ]:
import json, shutil, logging, zipfile
import pandas as pd
from IPython.display import Image, display
from src.utils.helpers import load_config, setup_logging

def find_dataset(name):
    """Accepts a full path (/kaggle/input/datasets/<user>/<name>), '<user>/<name>' or just '<name>'.
    Kaggle mounts datasets at /kaggle/input/<name> or /kaggle/input/datasets/<user>/<name>."""
    p = Path(name)
    if p.is_absolute() and p.is_dir():
        return p
    leaf = p.name
    hits = sorted((d for d in INPUT.rglob("*") if d.is_dir() and d.name == leaf), key=lambda d: len(d.parts))
    if hits:
        return hits[0]
    # last resort: the folder holding Planet surface-reflectance scenes, wherever it is mounted
    sr = sorted(INPUT.rglob("*AnalyticMS_SR*.tif"))
    return sr[0].parent if sr else None

IMAGERY = find_dataset(DATASET_NAME)
assert IMAGERY, (f"dataset {DATASET_NAME!r} not found under {INPUT}: attach it with Add Input. Found: "
                 f"{sorted(str(d.relative_to(INPUT)) for d in INPUT.rglob('*') if d.is_dir())[:20]}")
tifs = sorted(p for p in IMAGERY.rglob("*.tif") if "udm" not in p.name.lower())
print(f"{len(tifs)} scenes in {IMAGERY}")
assert tifs

setup_logging(WORK / "logs", "INFO", "kaggle")
QUICK = ["train.max_train_patches=48", "train.epochs=2", "evaluation.n_visuals=4"] if QUICK_TEST else []
REPORT = WORK / "best" / "report"; REPORT.mkdir(parents=True, exist_ok=True)

def make_cfg(prep, exp=None, extra=()):
    exp_dir = WORK / "experiments" / (exp or "_none")
    over = [f"paths.imagery_dir={IMAGERY}", f"paths.processed_dir={WORK / 'prep' / prep}",
            f"paths.models_dir={exp_dir / 'models'}", f"paths.outputs_dir={exp_dir / 'outputs'}",
            f"paths.logs_dir={WORK / 'logs'}", "train.num_workers=4", f"train.epochs={EPOCHS}",
            f"train.patience={PATIENCE}", *PREP[prep], *extra, *QUICK]
    return load_config(PROJECT / "configs" / "config.yaml", over)

def hours_left():
    return TIME_BUDGET_H - (time.time() - T0) / 3600

# --- backups: a zip in the output after every experiment, plus (optional) a private Kaggle dataset
TOKEN = None
if BACKUP_DATASET:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    except Exception as e:
        print("no KAGGLE_API_TOKEN secret: dataset backup off (the zip backup still works)", e)

def backup(note):
    stage = WORK / "backup_stage"; shutil.rmtree(stage, ignore_errors=True); stage.mkdir()
    for exp in EXPERIMENTS:
        mdir = WORK / "experiments" / exp / "models"
        for f in ("best_model.pth", "latest_checkpoint.pth", "training_history.csv"):
            if (mdir / f).exists():
                shutil.copy(mdir / f, stage / f"{exp}__{f}")   # flat names: no folders to unzip on restore
    with zipfile.ZipFile(WORK / "results_backup.zip", "w") as z:
        for f in stage.glob("*"):
            if not f.name.endswith("latest_checkpoint.pth"):
                z.write(f, f.name)
    if not TOKEN or not any(stage.iterdir()):
        return
    (stage / "dataset-metadata.json").write_text(json.dumps(
        {"title": BACKUP_DATASET, "id": f"{KAGGLE_USER}/{BACKUP_DATASET}", "licenses": [{"name": "other"}]}))
    env = {**os.environ, "KAGGLE_API_TOKEN": TOKEN}
    exists = subprocess.run(["kaggle", "datasets", "status", f"{KAGGLE_USER}/{BACKUP_DATASET}"], env=env,
                            capture_output=True, text=True).returncode == 0
    cmd = ["kaggle", "datasets", "version", "-p", str(stage), "-m", note] if exists else \
          ["kaggle", "datasets", "create", "-p", str(stage)]
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print("backup:", (r.stdout or r.stderr).strip().splitlines()[-1:] )

def restore(exp, mdir):
    """Continue from a previous run: this notebook's earlier output or the backup dataset, if attached."""
    if (mdir / "latest_checkpoint.pth").exists():
        return
    for pattern, flat in ((f"experiments/{exp}/models/latest_checkpoint.pth", False), (f"{exp}__latest_checkpoint.pth", True)):
        hits = sorted(INPUT.rglob(pattern))
        if hits:
            mdir.mkdir(parents=True, exist_ok=True)
            for f in ("best_model.pth", "latest_checkpoint.pth", "training_history.csv"):
                src_f = hits[0].parent / (f"{exp}__{f}" if flat else f)
                if src_f.exists():
                    shutil.copy(src_f, mdir / f)
            print(f"{exp}: restored from {hits[0].parent}")
            return

## 4. Preprocess

Masks → validation figures → geographic split (shared by both variants) → patches → normalisation → the 7
pre-training checks (a failure stops the notebook with the reason).

In [ ]:
from src.pipeline import prepare

first = next(iter(PREP))
for prep in PREP:
    cfg = make_cfg(prep)
    meta = Path(cfg.paths.processed_dir) / "metadata"
    if (meta / "patches.csv").exists() and (meta / "validation_report.json").exists():
        print(f"{prep}: already prepared"); continue
    if prep != first:
        meta.mkdir(parents=True, exist_ok=True)
        shutil.copy(WORK / "prep" / first / "metadata" / "split_blocks.json", meta / "split_blocks.json")
    print(f"\n========== preprocessing {prep} ==========")
    prepare(cfg)

patch_tables = {p: pd.read_csv(WORK / "prep" / p / "metadata" / "patches.csv") for p in PREP}
patch_summary = pd.concat({p: t.groupby("split").agg(patches=("patch_id", "size"), farms=("n_farms", "sum"),
                                                     farm_px=("farm_pixels", "sum"), boundary_px=("boundary_pixels", "sum"))
                           for p, t in patch_tables.items()})
rep = json.loads((WORK / "prep" / first / "metadata" / "preprocessing_report.json").read_text())
label_summary = pd.Series({k: rep[k] for k in ("source_polygons", "valid_polygons", "repaired",
                                               "removed_exact_duplicates", "removed_near_duplicates",
                                               "train_polygons", "ignored_polygons")}, name="farm polygons").to_frame()
checks = pd.DataFrame(json.loads((WORK / "prep" / first / "metadata" / "validation_report.json").read_text()))[["check", "ok", "message"]]
display(label_summary, patch_summary, checks)
mask_figs = sorted((WORK / "prep" / first / "visualizations").glob("*_masks.png"))
for f in mask_figs[:3] + [WORK / "prep" / first / "visualizations" / "split_map.png"]:
    display(Image(filename=str(f), width=1400))
print(f"{hours_left():.1f} h left")

## 5. Train every experiment (time-budgeted, backed up after each)

In [ ]:
from src.training.trainer import train
from src.utils.visualization import plot_history

best_pt = {}
todo = list(EXPERIMENTS)
for k, exp in enumerate(todo):
    prep, extra = EXPERIMENTS[exp]
    budget = (hours_left() - EVAL_RESERVE_H) / (len(todo) - k)
    if budget < 0.25 and not QUICK_TEST:
        print(f"{exp}: skipped - no time left (attach this version's output next time to continue)"); continue
    cfg = make_cfg(prep, exp, extra + [f"train.max_hours={max(budget, 0.25):.2f}"])
    mdir = Path(cfg.paths.models_dir)
    restore(exp, mdir)
    print(f"\n========== {exp} ({budget:.1f} h budget): {extra} ==========")
    best_pt[exp] = train(cfg)
    plot_history(mdir / "training_history.csv", mdir / "training_history.png")
    backup(f"after {exp}")
print(f"trained: {list(best_pt)} | {hours_left():.1f} h left")

### Training curves

In [ ]:
from src.utils.visualization import plot_training_curves
hist = {e: WORK / "experiments" / e / "models" / "training_history.csv" for e in best_pt}
curves_png = plot_training_curves(hist, REPORT / "training_curves.png")
display(Image(filename=str(curves_png), width=1500))
train_summary = pd.DataFrame({e: {"epochs run": len(pd.read_csv(h)),
                                  "best epoch": int(pd.read_csv(h).val_score.idxmax()),
                                  "best val_score": pd.read_csv(h).val_score.max(),
                                  "best boundary F1": pd.read_csv(h).boundary_f1.max(),
                                  "best interior IoU": pd.read_csv(h).interior_iou.max()} for e, h in hist.items()}).T
train_summary

## 6. Compare on the common validation reference

Every model predicts the validation patches of `REFERENCE_PREP` (hand-traced farms) once (with TTA), its
post-processing is tuned by the two-stage search, and it is scored on **individual farms**. Then the
`ENSEMBLE_TOP` best models are averaged and tuned the same way. The ensemble is used only if it beats the best
single model by more than 0.005 object F1 (it doubles mapping time).

In [ ]:
from src.training.evaluation import predict_patches, tune_postprocess, evaluate, ensemble_caches, true_areas
from src.utils.visualization import plot_comparison

ref_cfg = make_cfg(REFERENCE_PREP)
AREAS = true_areas(ref_cfg)
SMALL = dict(stage1={"seed_source": ["boundary", "distance"], "seed_threshold": [0.5], "min_seed_px": [25]},
             stage2={"interior_threshold": [0.35], "boundary_threshold": [0.5], "min_area_m2": [200]}) if QUICK_TEST else {}
caches, tuned, rows, grids = {}, {}, {}, {}

def score(name, cache):
    s1 = SMALL.get("stage1")
    if s1 and not cache["distance"]:
        s1 = {**s1, "seed_source": ["boundary"]}
    pc, grid = tune_postprocess(ref_cfg, cache, s1, SMALL.get("stage2"))
    tuned[name], grids[name] = pc, grid
    res = evaluate(ref_cfg, None, "val", WORK / "evaluation" / name / "val_reference", postprocess=pc, cache=cache,
                   areas=AREAS, title=name)
    rows[name] = {k: res[k] for k in ("object_f1", "object_precision", "object_recall", "mean_matched_iou",
                                      "merged_predictions", "split_farms", "interior_iou", "boundary_f1",
                                      "boundary_iou", "farms_true")}
    rows[name].update({f"pp_{k}": pc[k] for k in ("seed_source", "seed_threshold", "min_seed_px",
                                                  "interior_threshold", "boundary_threshold", "min_area_m2")})

for exp in best_pt:
    caches[exp] = predict_patches(ref_cfg, best_pt[exp], "val", tta=USE_TTA)
    score(exp, caches[exp])

singles = pd.DataFrame(rows).T.sort_values("object_f1", ascending=False)
ENS_MEMBERS = []
if ENSEMBLE_TOP > 1 and len(best_pt) > 1:
    ENS_MEMBERS = list(singles.index[:ENSEMBLE_TOP])
    score("ENSEMBLE", ensemble_caches([caches[e] for e in ENS_MEMBERS]))

comparison = pd.DataFrame(rows).T
comparison.to_csv(WORK / "comparison_val.csv")
order = [e for e in EXPERIMENTS if e in comparison.index] + (["ENSEMBLE"] if "ENSEMBLE" in comparison.index else [])
comparison = comparison.loc[order]
top_single = max((e for e in best_pt), key=lambda e: float(comparison.loc[e, "object_f1"]))
top_f1 = float(comparison.loc[top_single, "object_f1"])
BEST = next(e for e in best_pt if float(comparison.loc[e, "object_f1"]) >= top_f1 - 0.01)   # simplest within 0.01
if "ENSEMBLE" in comparison.index and float(comparison.loc["ENSEMBLE", "object_f1"]) > top_f1 + 0.005:
    BEST = "ENSEMBLE"
BEST_MODELS = [best_pt[e] for e in ENS_MEMBERS] if BEST == "ENSEMBLE" else best_pt[BEST]
print(f"BEST = {BEST}" + (f" ({ENS_MEMBERS})" if BEST == "ENSEMBLE" else ""),
      f"| val object F1 {float(comparison.loc[BEST, 'object_f1']):.3f}")
comp_png = plot_comparison(comparison, REPORT / "comparison_val.png")
display(Image(filename=str(comp_png), width=1200))
for c in caches.values():
    c["items"].clear()
comparison

### Validation by farm size and pixel confusion matrices (winner)

In [ ]:
vdir = WORK / "evaluation" / BEST / "val_reference"
display(pd.read_csv(vdir / "by_size.csv").set_index("size"))
display(Image(filename=str(vdir / "by_size.png"), width=1200))
display(Image(filename=str(vdir / "confusion.png"), width=1000))

## 7. Test the winner (held-out blocks, reported once)

In [ ]:
from src.utils.visualization import plot_object_outcomes
best_dir = WORK / "best"
pp = tuned[BEST]
cache_t = predict_patches(ref_cfg, BEST_MODELS, "test", tta=USE_TTA)
test_ref = evaluate(ref_cfg, None, "test", best_dir / "test_reference", postprocess=pp, cache=cache_t,
                    areas=AREAS, title=f"{BEST} (traced farms)")
own_prep = EXPERIMENTS[ENS_MEMBERS[0] if BEST == "ENSEMBLE" else BEST][0]
own_cfg = make_cfg(own_prep)
test_own = evaluate(own_cfg, None, "test", best_dir / "test_own_labels", postprocess=pp,
                    cache=predict_patches(own_cfg, BEST_MODELS, "test", tta=USE_TTA), areas=AREAS,
                    title=f"{BEST} (training labels)")
outcomes_png = plot_object_outcomes(test_ref, REPORT / "test_outcomes.png", f"{BEST}: test, hand-traced farms")

for m in (BEST_MODELS if isinstance(BEST_MODELS, list) else [BEST_MODELS]):
    shutil.copy(m, best_dir / f"{Path(m).parent.parent.name}_best_model.pth")
summary = {"best": BEST, "ensemble_members": ENS_MEMBERS if BEST == "ENSEMBLE" else [],
           "settings": {e: EXPERIMENTS[e] for e in (ENS_MEMBERS if BEST == "ENSEMBLE" else [BEST])},
           "tta": USE_TTA, "postprocess": dict(pp), "val_reference": rows[BEST],
           "test_reference": test_ref, "test_own_labels": test_own}
(best_dir / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
keys = ["object_f1", "object_precision", "object_recall", "mean_matched_iou", "merged_predictions", "split_farms",
        "interior_iou", "interior_dice", "interior_precision", "interior_recall", "boundary_iou", "boundary_f1",
        "boundary_precision", "boundary_recall", "farms_true"]
test_table = pd.DataFrame({"test (hand-traced farms)": {k: test_ref[k] for k in keys},
                           "test (training labels)": {k: test_own[k] for k in keys}})
display(test_table)
display(pd.read_csv(best_dir / "test_reference" / "by_size.csv").set_index("size"))
for f in (best_dir / "test_reference" / "by_size.png", best_dir / "test_reference" / "confusion.png", outcomes_png):
    display(Image(filename=str(f), width=1200))
for f in sorted((best_dir / "test_reference" / "visuals").glob("*.png"))[:4]:
    display(Image(filename=str(f), width=1500))

## 8. Map every scene with the winner

In [ ]:
from src.inference.predictor import Predictor
from src.gis.polygon_reconstruction import polygonize, export_polygons, merge_scenes

map_counts = {}
if MAP_ALL_SCENES:
    cfg_map = make_cfg(own_prep)
    cfg_map.paths.outputs_dir = best_dir / "map"
    cfg_map.postprocess = pp
    cfg_map.inference.tta = USE_TTA
    predictor = Predictor(BEST_MODELS, cfg_map)
    frames = []
    for img in tifs:
        prob = predictor.predict_raster(img)
        farms = polygonize(cfg_map, prob)
        export_polygons(farms, cfg_map.paths.outputs_dir, prob.stem.replace("_prob", "") + "_farms")
        frames.append(farms)
        map_counts[prob.stem.replace("_prob", "")] = len(farms)
    allf = merge_scenes(frames)
    paths = export_polygons(allf, cfg_map.paths.outputs_dir, "all_farms")
    map_counts["ALL (merged)"] = len(allf)
    print(f"{len(allf)} farms, {allf.area_ha.sum():.0f} ha; {int((allf.score < 0.6).sum())} low-confidence (orange)")
    print({k: str(v) for k, v in paths.items()})

In [ ]:
import numpy as np, rasterio, matplotlib.pyplot as plt
from rasterio.windows import from_bounds
from src.utils.visualization import stretch
preview_png = REPORT / "map_preview.png"
if MAP_ALL_SCENES and len(frames[0]):
    f0 = frames[0]; c = f0.geometry.centroid
    cell = (np.floor(c.x / 3000) * 3000).astype(int).astype(str) + "_" + (np.floor(c.y / 3000) * 3000).astype(int).astype(str)
    x0, y0 = map(float, cell.value_counts().index[0].split("_")); b = (x0, y0, x0 + 3000, y0 + 3000)
    with rasterio.open(tifs[0]) as s:
        rgb = s.read([6, 4, 2] if s.count == 8 else [3, 2, 1], window=from_bounds(*b, s.transform))
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(stretch(rgb), extent=(b[0], b[2], b[1], b[3]))
    f0.cx[b[0]:b[2], b[1]:b[3]].boundary.plot(ax=ax, color="yellow", linewidth=0.8)
    ax.set_xlim(b[0], b[2]); ax.set_ylim(b[1], b[3]); ax.set_title(f"{BEST}: predicted farms, 3 x 3 km"); ax.axis("off")
    fig.savefig(preview_png, dpi=100, bbox_inches="tight"); plt.show()

## 9. Report (one HTML file with every table and figure) and final backup

In [ ]:
from src.utils.report import build_report
vt = WORK / "evaluation" / BEST / "val_reference"
tt = best_dir / "test_reference"
report = build_report(best_dir / "report.html", "Farm boundary detection - round 2 results", [
    ("Summary", [f"Winner: <b>{BEST}</b>" + (f" (ensemble of {', '.join(ENS_MEMBERS)})" if BEST == "ENSEMBLE" else "")
                 + f"; TTA {'on' if USE_TTA else 'off'}. Test object F1 on hand-traced farms: "
                 f"<b>{test_ref['object_f1']:.3f}</b> (precision {test_ref['object_precision']:.3f}, recall "
                 f"{test_ref['object_recall']:.3f}). Round 1 (validation): 0.282.", test_table]),
    ("Data", [label_summary, patch_summary, checks] + mask_figs[:2]),
    ("Training curves", [train_summary, curves_png]),
    ("Validation comparison (common reference: hand-traced farms)", [comparison, comp_png]),
    ("Winner on validation: by farm size and confusion matrices", [pd.read_csv(vt / "by_size.csv").set_index("size"),
                                                                    vt / "by_size.png", vt / "confusion.png"]),
    ("Winner on test (held out)", [pd.read_csv(tt / "by_size.csv").set_index("size"), tt / "by_size.png",
                                   tt / "confusion.png", outcomes_png] + sorted((tt / "visuals").glob("*.png"))[:3]),
    ("Map", [pd.Series(map_counts, name="farms").to_frame()] + ([preview_png] if preview_png.exists() else [])),
])
print("report ->", report)
backup("final")
with zipfile.ZipFile(WORK / "best_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in best_dir.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(WORK))
print("everything in best/ zipped -> best_results.zip;", f"{hours_left():.1f} h of budget left")

## 10. Download

From the version's **Output** tab:

* `best/report.html`: the full report (open in a browser) - data checks, training curves, comparison,
  confusion matrices, scores by farm size, test results, map preview;
* `best/map/kmz/all_farms.kmz` (Google Earth Pro; orange = review first) and `best/map/geojson|shapefiles|gpkg/`
  (QGIS);
* `best/summary.json` (winner, tuned thresholds, scores) and `best/*_best_model.pth`;
* `best_results.zip` (all of `best/`), `results_backup.zip` (all models), `comparison_val.csv`.